In [1]:
import pandas as pd
import re

df = pd.read_csv('/kaggle/input/datasets/danielantoniudumitru/spanish-dataset/job_applicant_dataset_es_fixed.csv')
print(f"Loaded {len(df)} rows")


def split_bullets(text):
    text = text.strip().lstrip('- ').strip()
    items = re.split(r'\s+-\s+', text)
    cleaned = []
    for i in items:
        i = re.sub(r'\s*-\s*$', '', i).strip()  # remove trailing dash
        if i and len(i) > 1:
            cleaned.append(i)
    return cleaned

def as_bullets(text):
    return '\n'.join(f'- {i}' for i in split_bullets(text))


JD_SECS = [
    ('about',  r'(?:Acerca del [Rr]ol|Acerca del [Pp]apel|Sobre el [Rr]ol|Sobre el [Pp]apel)'),
    ('skills', r'(?:Habilidades [Rr]equeridas?|Skills [Rr]equeridos?|Habilidades [Nn]ecesarias?)'),
    ('resp',   r'Responsabilidades'),
    ('offer',  r'(?:Lo que [Oo]frecemos|Qué [Oo]frecemos)'),
    ('apply',  r'(?:Cómo [Aa]plicar|Cómo [Ss]olicitar)'),
]

def reformat_jd(text, job_role):
    if not text or str(text).strip() == 'nan':
        return str(text)
    text = str(text).strip()
    role = str(job_role).strip()

    secs = {}
    for key, pat in JD_SECS:
        m = re.search(pat, text)
        if m:
            secs[key] = (m.start(), m.end())

    intro_start = 0
    rm = re.match(re.escape(role) + r'\s*', text, re.IGNORECASE)
    if rm:
        intro_start = rm.end()
    first_sec_start = min((v[0] for v in secs.values()), default=len(text))
    intro = text[intro_start:first_sec_start].strip()

    def extract(key):
        if key not in secs:
            return ''
        s_start, s_end = secs[key]
        nexts = sorted([v[0] for k, v in secs.items() if k != key and v[0] > s_start])
        content_end = nexts[0] if nexts else len(text)
        return text[s_end:content_end].strip()

    out = [role, '']
    if intro:
        out += [intro, '']
    about = extract('about')
    if about:
        out += ['Acerca del Rol', about, '']
    skills = extract('skills')
    if skills:
        out += ['Habilidades Requeridas', as_bullets(skills), '']
    resp = extract('resp')
    if resp:
        out += ['Responsabilidades', as_bullets(resp), '']
    offer = extract('offer')
    if offer:
        out += ['Lo que Ofrecemos', as_bullets(offer), '']
    apply_t = extract('apply')
    if apply_t:
        out += ['Cómo Aplicar', apply_t.lstrip('- ').strip()]

    return '\n'.join(out).strip()


R_SECS = [
    ('summary', r'(?:RESUMEN\s+PROFESSIONAL|RESUMEN\s+PROFESIONAL|PROFESSIONAL\s+SUMMARY)'),
    ('exp',     r'(?:PROFESSIONAL\s+EXPERIENCE|EXPERIENCIA\s+PROFESIONAL|EXPERIENCIA\s+LABORAL)'),
    ('edu',     r'(?:EDUCACI[ÓO]N|EDUCATION|FORMACI[ÓO]N)\b'),
    ('skills',  r'(?:HABILIDADES|SKILLS)\b'),
    ('certs',   r'(?:CERTIFICACIONES|CERTIFICATIONS)\b'),
    ('langs',   r'(?:IDIOMAS|LANGUAGES|LANGUADES)\b'),
]

def reformat_resume(text, name):
    if not text or str(text).strip() == 'nan':
        return str(text)
    text = str(text).strip()
    name_str = str(name).strip()

    secs = {}
    for key, pat in R_SECS:
        m = re.search(pat, text)
        if m:
            secs[key] = (m.start(), m.end())

    name_pos = text.find(name_str)
    if name_pos == -1:
        for p in reversed(name_str.split()):
            if len(p) > 2:
                idx = text.find(p)
                if idx != -1:
                    name_pos = idx
                    break
    if name_pos == -1:
        name_pos = 0

    first_sec = min((v[0] for v in secs.values()), default=len(text))
    dash_m = re.search(r'---', text)
    if dash_m and dash_m.start() < first_sec:
        first_sec = dash_m.start()

    contact_raw = text[name_pos:first_sec].strip()

    email_m    = re.search(r'\S+@\S+\.\S+', contact_raw)
    linkedin_m = re.search(r'https?://www\.linkedin\.com/\S+', contact_raw)
    phone_m    = re.search(r'\+\d[\d\s\-]{6,}', contact_raw)

    addr_clean = contact_raw
    addr_clean = addr_clean.replace(name_str, '', 1)
    for m in [email_m, linkedin_m, phone_m]:
        if m:
            addr_clean = addr_clean.replace(m.group(), '')
    addr_clean = re.sub(r'\s+', ' ', addr_clean).strip()

    addr_m = re.match(r'(\d+\s+.+?)\s+([A-ZÁÉÍÓÚÑ][^,\d]+(?:,\s*[^,\d]+)+)', addr_clean)
    if addr_m:
        street = addr_m.group(1).strip().rstrip(',')
        city   = addr_m.group(2).strip()
    else:
        parts  = addr_clean.split(',')
        street = parts[0].strip() if parts else addr_clean
        city   = ', '.join(p.strip() for p in parts[1:]) if len(parts) > 1 else ''

    contact_lines = [name_str]
    if street:
        contact_lines.append(street)
    if city:
        contact_lines.append(city)
    if phone_m:
        contact_lines.append(phone_m.group().strip())
    if email_m:
        contact_lines.append(email_m.group().strip())
    if linkedin_m:
        contact_lines.append(linkedin_m.group().strip())

    def extract(key):
        if key not in secs:
            return ''
        s_start, s_end = secs[key]
        nexts = sorted([v[0] for k, v in secs.items() if k != key and v[0] > s_start])
        content_end = nexts[0] if nexts else len(text)
        raw = text[s_end:content_end]
        raw = re.sub(r'^\s*-{2,}\s*', '', raw)
        return raw.strip()

    summary_text = extract('summary')
    exp_text     = extract('exp')
    edu_text     = extract('edu')
    skills_text  = extract('skills')
    certs_text   = extract('certs')
    langs_text   = extract('langs')

    def clean_summary(t):
        t = re.sub(r'\s*-{2,}\s*$', '', t).strip()
        return t

    def fix_exp_bullets(t):
        t = re.sub(r'([^\n])\s*-\s+(?=[A-ZÁÉÍÓÚÑ\w])', r'\1\n- ', t)
        t = re.sub(r'\n{3,}', '\n\n', t)
        return t.strip()

    def clean_skill_list(t):
        for pat in [r'CERTIFICACIONES.*', r'IDIOMAS.*', r'LANGUADES.*', r'LANGUAGES.*']:
            t = re.sub(pat, '', t, flags=re.DOTALL | re.IGNORECASE)
        items = re.split(r'\s*-\s+', t.strip())
        items = [re.sub(r'\s*-\s*$', '', i).strip() for i in items]
        items = [i for i in items if i and len(i) > 1]
        return '\n'.join(f'- {i}' for i in items)

    def clean_cert_list(t):
        for pat in [r'(?:IDIOMAS|LANGUAGES|LANGUADES).*']:
            t = re.sub(pat, '', t, flags=re.DOTALL | re.IGNORECASE)
        items = re.split(r'\s*-\s+', t.strip())
        items = [re.sub(r'\s*-\s*$', '', i).strip() for i in items]
        items = [i for i in items if i and len(i) > 1]
        return '\n'.join(f'- {i}' for i in items)

    def clean_lang_list(t):
        items = re.split(r'\s*-\s+', t.strip())
        items = [re.sub(r'\s*-\s*$', '', i).strip() for i in items]
        items = [i for i in items if i and len(i) > 1]
        return '\n'.join(f'- {i}' for i in items)

    out = ['\n'.join(contact_lines), '', '---']

    if summary_text:
        out += ['', 'RESUMEN PROFESIONAL', '', clean_summary(summary_text), '', '---']
    if exp_text:
        out += ['', 'EXPERIENCIA PROFESIONAL', '', fix_exp_bullets(exp_text), '', '---']
    if edu_text:
        out += ['', 'EDUCACIÓN', '', edu_text.strip(), '', '---']
    if skills_text:
        sl = clean_skill_list(skills_text)
        if sl:
            out += ['', 'HABILIDADES', '', sl]
    if certs_text:
        cl = clean_cert_list(certs_text)
        if cl:
            out += ['', 'CERTIFICACIONES', '', cl]
    if langs_text:
        ll = clean_lang_list(langs_text)
        if ll:
            out += ['', 'IDIOMAS', '', ll]

    return '\n'.join(out).strip()


print("Reformatting Job Descriptions...")
df['Job Description'] = [reformat_jd(r['Job Description'], r['Job Roles']) for _, r in df.iterrows()]
print("Reformatting Resumes...")
df['Resume'] = [reformat_resume(r['Resume'], r['Job Applicant Name']) for _, r in df.iterrows()]

out_path = '/kaggle/working/job_applicant_dataset_es_formatted.csv'
df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Saved {len(df)} rows → {out_path}")

print("\n=== JD row 0 ===")
print(df['Job Description'].iloc[0])
print("\n=== Resume row 0 ===")
print(df['Resume'].iloc[0])
print("\n=== JD row 1 ===")
print(df['Job Description'].iloc[1])
print("\n=== Resume row 1 ===")
print(df['Resume'].iloc[1])

Loaded 10000 rows
Reformatting Job Descriptions...
Reformatting Resumes...
Saved 10000 rows → /kaggle/working/job_applicant_dataset_es_formatted.csv

=== JD row 0 ===
Entrenador de fitness

Fitness Coach A Fitness Coach es responsable de ayudar a los clientes a alcanzar sus objetivos de fitness diseñando y liderando programas de fitness individuales o grupales. Usted proporcionará instrucción sobre ejercicios, forma adecuada y técnicas de prevención de lesiones, animando a los clientes a empujar sus límites mientras mantienen un enfoque en su bienestar.

Acerca del Rol
El papel requiere una pasión por la salud y la aptitud física, una comprensión sólida de la fisiología del ejercicio y la capacidad de motivar e inspirar a otros. También supervisará el progreso de los clientes y realizar ajustes en sus planes de fitness según sea necesario para garantizar una mejora continua.

Habilidades Requeridas
- Prevención de lesiones
- Motivación
- Nutrición
- Entrenamiento en Salud

Responsabili